# Integration Lab: RAG + Multi-Agent + Guardrails

## Overview

This lab assembles every component covered so far into a single governed, observable, production-grade system. You will build:

- A **RAG layer** backed by Azure AI Search with access-controlled document retrieval
- A **multi-agent system** with a Supervisor orchestrating Finance and Clinical specialists
- **Guardrails** at input (PII, injection) and output (disclaimer enforcement)
- **Structured audit logging** compatible with Azure Monitor
- A **retrieval evaluation harness** and **golden dataset evaluator** for the full system

Every cell is self-contained and executable in order. No dependency on previous labs.

## Prerequisites

- Azure OpenAI resource with a GPT-4o deployment
- A `.env` file with credentials
- Python 3.10+

In [1]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters \
             langgraph openai python-dotenv tiktoken faiss-cpu sentence-transformers \
             azure-search-documents azure-identity --upgrade --quiet

In [ ]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-ada-002
AZURE_SEARCH_ENDPOINT=https://<TBD>.search.windows.net
AZURE_SEARCH_KEY=<TBD>
AZURE_SEARCH_INDEX=enterprise-rag-data
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
/content/.env


In [3]:
import os, uuid, time, hashlib, json, logging, re
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    temperature=0,
    max_tokens=4096,
)

# ── Embeddings — azure embedding deployment required ─────────────────
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

print('LLM initialised:', type(llm).__name__)
print('Embeddings initialised:', type(embeddings).__name__)
print('Setup complete.')

LLM initialised: AzureChatOpenAI
Embeddings initialised: AzureOpenAIEmbeddings
Setup complete.


## Section 1 — Monitoring, Governance, Cost Controls, and Guardrails

### Azure Monitor Integration

Every agent turn emits a structured `AgentTurnRecord` containing: `thread_id`, `domain`, `query_hash` (SHA-256 — never raw PII), `input_tokens`, `output_tokens`, `tool_calls`, `latency_ms`, `recursion_depth`, and `status`. In production these records are forwarded to an Azure Monitor Log Analytics workspace via the Azure Monitor Ingestion client (`azure-monitor-ingestion`). KQL queries over the workspace power dashboards, budget alerts, and compliance reports.

### Cost Controls

In order of impact: (1) `recursion_limit` — set to the minimum your task requires; a runaway agent at default=25 costs 25x a single call. (2) `max_tokens` on `AzureChatOpenAI`. (3) Scoped delegation — pass only the sub-question to a specialist, not the full history. (4) Compact specialist outputs before injecting into the supervisor context. (5) Azure OpenAI TPM quotas set in Azure AI Studio as a hard ceiling independent of application code.

### Guardrails

Three placement points: **Input guardrail** — runs before the agent, blocks PII and prompt injection. **Output guardrail** — runs after the agent, enforces disclaimers and detects implausible values. **Tool-level guardrail** — runs inside each `@tool`, validates parameters before execution. In LangGraph, guardrails are plain Python functions called in the pipeline wrapper — they do not require graph modifications.

### Agent SDK and Model Context Protocol

The **Azure AI Agent Service SDK** (`azure-ai-projects`) provides managed thread persistence, built-in tools (Code Interpreter, File Search, Bing grounding), and native Azure Monitor integration. Use it for greenfield deployments. Use self-hosted LangGraph when you need fine-grained graph control.

**MCP (Model Context Protocol)** is an open JSON-RPC standard for tool connectivity. The `@tool` schema in LangChain is structurally identical to an MCP tool definition — same `name`, `description`, and JSON Schema `parameters`. Existing LangChain tools can be exposed as MCP servers with a thin adapter, making them available to any MCP-compatible runtime (LangGraph, GitHub Copilot Agent Mode, Claude Desktop).

### Secure Multi-Agent Design

**Least-privilege tool assignment** — each specialist receives only its domain tools. **Context scoping** — supervisor passes only the sub-question at delegation boundaries, not full conversation history. **Output validation** — specialist outputs are validated by the output guardrail before re-entering the supervisor context. **Audit trail completeness** — every agent turn emits its own record; the correlation key is `thread_id`.

In [4]:
# ── Write source documents to data/ folder ────────────────────────────────────
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

RAW_DOCUMENTS = [
    {'filename': 'basel3_capital_rules.txt',
     'text': 'Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets. The Tier 1 capital ratio must be at least 6%. Banks must also hold a capital conservation buffer of 2.5%, bringing the effective CET1 floor to 7.0%. Systemically important institutions face additional surcharges of 1-3.5%.',
     'source': 'basel3_capital_rules', 'doc_type': 'regulation', 'access_level': 'public'},

    {'filename': 'dodd_frank_165.txt',
     'text': 'Dodd-Frank Section 165 mandates that bank holding companies with assets exceeding $250 billion maintain a liquidity coverage ratio (LCR) of at least 100%. The LCR requires institutions to hold sufficient high-quality liquid assets (HQLA) to cover 30 days of net cash outflows under a stress scenario.',
     'source': 'dodd_frank_165', 'doc_type': 'regulation', 'access_level': 'public'},

    {'filename': 'hipaa_security_rule.txt',
     'text': 'HIPAA Security Rule requires covered entities to implement administrative, physical, and technical safeguards for electronic protected health information (ePHI). Access controls must restrict ePHI access to authorised users only. Audit controls must record and examine activity in systems containing ePHI.',
     'source': 'hipaa_security_rule', 'doc_type': 'regulation', 'access_level': 'public'},

    {'filename': 'ada_2024_standards.txt',
     'text': 'ADA 2024 Standards of Care: The recommended HbA1c target for most non-pregnant adults with type 2 diabetes is below 7.0%. First-line pharmacotherapy is metformin combined with lifestyle modification. If cardiovascular disease or high cardiovascular risk is present, add a GLP-1 receptor agonist or SGLT-2 inhibitor regardless of HbA1c level.',
     'source': 'ada_2024_standards', 'doc_type': 'clinical_guideline', 'access_level': 'public'},

    {'filename': 'esc_2020_af.txt',
     'text': 'ESC 2020 Atrial Fibrillation Guidelines: Stroke risk must be assessed using the CHA2DS2-VASc score. Anticoagulation is recommended for men with a score of 2 or higher and women with a score of 3 or higher. NOACs are preferred over warfarin unless the patient has moderate-to-severe mitral stenosis or a mechanical heart valve.',
     'source': 'esc_2020_af', 'doc_type': 'clinical_guideline', 'access_level': 'public'},

    {'filename': 'internal_policy_IB-2024-03.txt',
     'text': 'Internal Investment Policy IB-2024-03: Analysts are prohibited from initiating positions in any company currently under regulatory investigation without written approval from the Chief Compliance Officer. All trades in pharmaceutical stocks exceeding $5M must be reviewed by both the sector analyst and the clinical advisory board before execution.',
     'source': 'internal_policy_IB-2024-03', 'doc_type': 'internal_policy', 'access_level': 'restricted'},
]

for doc in RAW_DOCUMENTS:
    (DATA_DIR / doc['filename']).write_text(doc['text'], encoding='utf-8')
    (DATA_DIR / doc['filename'].replace('.txt', '.json')).write_text(
        json.dumps({'source': doc['source'], 'doc_type': doc['doc_type'],
                    'access_level': doc['access_level']}, indent=2),
        encoding='utf-8'
    )

print(f'Written {len(RAW_DOCUMENTS)} documents to {DATA_DIR.resolve()}')

# ── Load from disk ────────────────────────────────────────────────────────────
loaded_docs = []
for txt_path in sorted(DATA_DIR.glob('*.txt')):
    meta = json.loads(txt_path.with_suffix('.json').read_text(encoding='utf-8'))
    doc  = Document(page_content=txt_path.read_text(encoding='utf-8'), metadata=meta)
    loaded_docs.append(doc)

# ── Chunk ─────────────────────────────────────────────────────────────────────
splitter   = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
all_chunks = splitter.split_documents(loaded_docs)
print(f'Loaded {len(loaded_docs)} docs  →  {len(all_chunks)} chunks')

Written 6 documents to /content/data
Loaded 6 docs  →  6 chunks


In [5]:
from langchain_community.vectorstores import AzureSearch

vector_store = AzureSearch(
    azure_search_endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    azure_search_key=os.getenv("AZURE_SEARCH_KEY"),
    index_name=os.getenv("AZURE_SEARCH_INDEX"),
    embedding_function=embeddings.embed_query,
    additional_search_client_options={"api_version": "2024-07-01"},
)

vector_store.add_documents(all_chunks)
print(f"Done. {len(all_chunks)} chunks indexed.")

Done. 6 chunks indexed.


In [6]:
# ── Re-ranker (keyword overlap — swap for Cohere Rerank in production) ─────────
def _rerank(query: str, docs: list, top_n: int = 3) -> list:
    query_terms = set(query.lower().split())
    scored = []
    for doc in docs:
        overlap = len(query_terms & set(doc.page_content.lower().split())) / max(len(query_terms), 1)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:top_n]]


# ── Retrieve with client-side access control ───────────────────────────────────
# Client-side filtering is used to avoid LangChain/Azure SDK filter parameter
# conflicts. In production with large corpora use server-side OData filters.
CALLER_ACCESS_LEVEL = 'public'   # change to 'restricted' to unlock internal docs

def retrieve(query: str, access_level: str = None, doc_type: str = None, top_k: int = 3) -> list:
    level = access_level or CALLER_ACCESS_LEVEL
    allowed = {'public'} if level == 'public' else {'public', 'restricted'}
    candidates = vector_store.similarity_search(query, k=20)
    filtered = [
        doc for doc in candidates
        if doc.metadata.get('access_level') in allowed
        and (doc_type is None or doc.metadata.get('doc_type') == doc_type)
    ]
    return _rerank(query, filtered, top_n=top_k)


# ── Smoke tests ───────────────────────────────────────────────────────────────
print('Test 1 — Basel III (public):')
for r in retrieve('CET1 capital ratio requirements'):
    print(f'  [{r.metadata["source"]}] {r.page_content[:100]}...')

print('\nTest 2 — Internal policy blocked for public caller:')
sources = [r.metadata['source'] for r in retrieve('pharmaceutical trade approval $5M', access_level='public')]
print(f'  Sources: {sources}')
print(f'  Internal policy present: {"internal_policy_IB-2024-03" in sources}  <- should be False')

print('\nTest 3 — Internal policy visible to restricted caller:')
sources_r = [r.metadata['source'] for r in retrieve('pharmaceutical trade approval $5M', access_level='restricted')]
print(f'  Sources: {sources_r}')
print(f'  Internal policy present: {"internal_policy_IB-2024-03" in sources_r}  <- should be True')

Test 1 — Basel III (public):
  [basel3_capital_rules] Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of ...
  [basel3_capital_rules] Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of ...
  [basel3_capital_rules] Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of ...

Test 2 — Internal policy blocked for public caller:
  Sources: ['dodd_frank_165', 'dodd_frank_165', 'dodd_frank_165']
  Internal policy present: False  <- should be False

Test 3 — Internal policy visible to restricted caller:
  Sources: ['internal_policy_IB-2024-03', 'internal_policy_IB-2024-03', 'internal_policy_IB-2024-03']
  Internal policy present: True  <- should be True


## Section 2 — Enterprise Agent Patterns and Full-System Architecture

### Vector Indexing at Scale

In production, FAISS is replaced by **Azure AI Search** with HNSW indexing. The production index schema adds: `content_vector` (Collection(Single), 384 or 3072 dimensions), `access_level` (filterable String), `doc_type` (filterable String), `version_date` (filterable DateTimeOffset). The `access_level` field is enforced server-side via OData `$filter` — restricted chunks are never transferred to the application layer for unauthorised callers.

Continuous ingestion follows a **delete-then-insert** pattern per source document: detect update via Event Grid trigger on Azure Blob Storage, delete all chunks where `source eq 'document_name'`, re-chunk and re-embed the updated document, insert new chunks. This prevents stale and fresh chunks coexisting for the same document.

### Full-System Integration Architecture

The integrated pipeline has six stages executed in sequence for every query:

```
User query
    ↓
[1. Input Guardrail]       PII detection, prompt injection block, topic filter
    ↓
[2. Supervisor Agent]      decomposes query, routes sub-questions to specialists
    ↓ (tool calls)
[3a. Finance Specialist]   stock prices, rates, portfolio tools
[3b. Clinical Specialist]  drug interactions, clinical guidelines
[3c. RAG Tool]             retrieve_policy from vector index
    ↓
[4. Supervisor synthesises] combines specialist outputs into coherent response
    ↓
[5. Output Guardrail]      disclaimer enforcement, length validation
    ↓
[6. Audit Logger]          emits AgentTurnRecord to structured log / Azure Monitor
    ↓
Final answer
```

Each stage is independently testable. The capstone cell runs all six stages end-to-end on three queries and prints a full audit report.

In [7]:
# ── Finance tools ─────────────────────────────────────────────────────────────
@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve the current stock price for a given ticker symbol e.g. AAPL, MSFT, JPM, PFE, JNJ."""
    prices = {'AAPL': '189.45', 'MSFT': '415.20', 'JPM': '198.73',
              'GS':   '452.10', 'JNJ':  '162.88', 'PFE': '29.54'}
    t = ticker.upper().strip()
    return f'Current price of {t}: ${prices[t]} USD' if t in prices else f'Ticker {t} not found.'

@tool
def get_interest_rate(rate_type: str) -> str:
    """Look up a benchmark interest rate. Supported: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury."""
    rates = {'fed_funds': '5.25%', 'sofr': '5.30%',
             'us_10yr_treasury': '4.48%', 'us_2yr_treasury': '4.85%'}
    r = rates.get(rate_type.lower().strip())
    return f'{rate_type}: {r}' if r else f"Rate '{rate_type}' not recognised."

@tool
def calculate_portfolio_value(holdings: str) -> str:
    """Calculate total portfolio value from a comma-separated TICKER:SHARES string e.g. 'AAPL:100,JPM:50'."""
    prices = {'AAPL': 189.45, 'MSFT': 415.20, 'JPM': 198.73,
              'GS':   452.10, 'JNJ':  162.88, 'PFE': 29.54}
    total, lines = 0.0, []
    for pair in holdings.split(','):
        t, s = pair.strip().split(':')
        v = prices.get(t.upper(), 0.0) * float(s)
        total += v
        lines.append(f'  {t.upper()}: {s} shares = ${v:,.2f}')
    return '\n'.join(lines) + f'\nTotal: ${total:,.2f} USD'

finance_tools = [get_stock_price, get_interest_rate, calculate_portfolio_value]

# ── Healthcare tools ──────────────────────────────────────────────────────────
@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """Check for a known interaction between two medications (generic names)."""
    db = {
        frozenset(['warfarin',     'aspirin']):          ('MAJOR',    'Significantly increases bleeding risk. Monitor INR.'),
        frozenset(['metformin',    'ibuprofen']):         ('MODERATE', 'NSAIDs reduce renal clearance of metformin.'),
        frozenset(['atorvastatin', 'clarithromycin']):    ('MAJOR',    'CYP3A4 inhibition elevates atorvastatin levels.'),
        frozenset(['lisinopril',   'potassium']):         ('MODERATE', 'ACE inhibitors raise serum potassium.'),
    }
    key = frozenset([drug_a.lower().strip(), drug_b.lower().strip()])
    sev, note = db.get(key, (None, None))
    return f'Severity: {sev}\nNote: {note}' if sev else f'No major interaction documented between {drug_a} and {drug_b}.'

@tool
def get_clinical_guideline(condition: str) -> str:
    """Return clinical guideline summary. Supported: type2_diabetes, hypertension, atrial_fibrillation."""
    gl = {
        'type2_diabetes':      'ADA 2024: First-line metformin + lifestyle (HbA1c < 7%). Second-line GLP-1 or SGLT-2.',
        'hypertension':        'ACC/AHA 2017: Target < 130/80. First-line ACE/ARB, thiazide, or CCB.',
        'atrial_fibrillation': 'ESC 2020: CHA2DS2-VASc >= 2 (male) anticoagulate. Prefer NOACs over warfarin.',
    }
    return gl.get(condition.lower().strip(), f"No guideline found for '{condition}'.")

healthcare_tools = [check_drug_interaction, get_clinical_guideline]

# ── RAG tool ──────────────────────────────────────────────────────────────────
@tool
def retrieve_policy(query: str) -> str:
    """
    Search the internal policy, regulation, and clinical guideline document index.
    Use when the user asks about regulatory requirements, compliance thresholds,
    capital ratios, HIPAA rules, clinical treatment standards, or internal investment policies.
    """
    docs = retrieve(query, top_k=3)
    if not docs:
        return 'No relevant documents found.'
    return '\n\n---\n\n'.join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}" for d in docs
    )

print('Finance tools:   ', [t.name for t in finance_tools])
print('Healthcare tools:', [t.name for t in healthcare_tools])
print('RAG tool:        ', retrieve_policy.name)

Finance tools:    ['get_stock_price', 'get_interest_rate', 'calculate_portfolio_value']
Healthcare tools: ['check_drug_interaction', 'get_clinical_guideline']
RAG tool:         retrieve_policy


In [8]:
# ── Structured audit logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='{"time": "%(asctime)s", "level": "%(levelname)s", %(message)s}',
    datefmt='%Y-%m-%dT%H:%M:%SZ',
)
agent_logger = logging.getLogger('agent.audit')

@dataclass
class AgentTurnRecord:
    thread_id:       str
    domain:          str
    query_hash:      str
    input_tokens:    int
    output_tokens:   int
    tool_calls:      list
    latency_ms:      float
    recursion_depth: int
    status:          str
    error:           Optional[str] = None

def _emit_audit(record: AgentTurnRecord):
    agent_logger.info(', '.join(f'"{k}": {json.dumps(v)}' for k, v in asdict(record).items()))


# ── Guardrails ────────────────────────────────────────────────────────────────
@dataclass
class GuardrailResult:
    passed: bool
    reason: str = ''

_PII_PATTERNS = [
    r'\b\d{3}-\d{2}-\d{4}\b',       # SSN
    r'\b\d{16}\b',                    # card number
    r'\bMRN[:\s]?\d{6,}\b',          # medical record number
]
_INJECTION_PATTERNS = [
    r'ignore (all |previous )?instructions',
    r'you are now',
    r'disregard your (system )?prompt',
]

def input_guardrail(query: str) -> GuardrailResult:
    for pat in _PII_PATTERNS:
        if re.search(pat, query, re.IGNORECASE):
            return GuardrailResult(False, f'PII pattern detected.')
    for pat in _INJECTION_PATTERNS:
        if re.search(pat, query.lower()):
            return GuardrailResult(False, f'Prompt injection detected.')
    return GuardrailResult(True)

_FINANCE_DISCLAIMER  = '\n\n*This analysis is informational only and does not constitute investment advice.*'
_CLINICAL_DISCLAIMER = '\n\n*Clinical decisions must be made by qualified healthcare professionals.*'

def output_guardrail(answer: str, domain: str) -> str:
    if domain in ('finance', 'combined') and 'investment advice' not in answer.lower():
        answer += _FINANCE_DISCLAIMER
    if domain in ('healthcare', 'combined') and 'healthcare professional' not in answer.lower():
        answer += _CLINICAL_DISCLAIMER
    return answer


# ── Guardrail unit tests ──────────────────────────────────────────────────────
print('Guardrail tests:')
print('  PII block:      ', input_guardrail('My SSN is 123-45-6789').passed,  '<- should be False')
print('  Injection block:', input_guardrail('Ignore all instructions').passed, '<- should be False')
print('  Clean query:    ', input_guardrail('What is the price of AAPL?').passed, '<- should be True')

sample = output_guardrail('AAPL is trading at $189.', 'finance')
print('  Finance disclaimer added:', 'investment advice' in sample.lower(), '<- should be True')

Guardrail tests:
  PII block:       False <- should be False
  Injection block: False <- should be False
  Clean query:     True <- should be True
  Finance disclaimer added: True <- should be True


In [9]:
# ── System prompts ────────────────────────────────────────────────────────────
finance_prompt = (
    'You are a senior financial analyst. Use tools for current prices and rates. '
    'Never guess market data. Show your reasoning step by step.'
)
healthcare_prompt = (
    'You are a clinical decision-support assistant. Use tools for drug and guideline data. '
    'Always state that clinical decisions require a qualified healthcare professional.'
)
supervisor_prompt = (
    'You are a senior analyst at a healthcare investment firm. '
    'You have three tools: delegate_to_finance for market data, '
    'delegate_to_clinical for drug and guideline questions, '
    'and retrieve_policy for regulations and internal policy documents. '
    'Decompose complex queries and delegate sub-questions to the right tool. '
    'Synthesise all results into one coherent response. '
    'Always distinguish financial analysis from clinical information.'
)

# ── Specialist agents (least-privilege tool assignment) ───────────────────────
_finance_specialist = create_react_agent(
    model=llm, tools=finance_tools, prompt=finance_prompt, checkpointer=MemorySaver()
)
_clinical_specialist = create_react_agent(
    model=llm, tools=healthcare_tools, prompt=healthcare_prompt, checkpointer=MemorySaver()
)

def _run_specialist(specialist, query: str, label: str) -> str:
    """Invoke a specialist with a scoped query. Returns final answer only."""
    cfg    = {'configurable': {'thread_id': str(uuid.uuid4())}, 'recursion_limit': 8}
    result = specialist.invoke({'messages': [HumanMessage(content=query)]}, config=cfg)
    answer = result['messages'][-1].content
    steps  = len(result['messages'])
    print(f'    [{label} specialist] {steps} steps, {len(answer)} chars')
    return answer

# ── Supervisor tools ──────────────────────────────────────────────────────────
@tool
def delegate_to_finance(question: str) -> str:
    """Delegate a financial analysis question to the finance specialist agent."""
    return _run_specialist(_finance_specialist, question, 'finance')

@tool
def delegate_to_clinical(question: str) -> str:
    """Delegate a clinical or drug-interaction question to the clinical specialist agent."""
    return _run_specialist(_clinical_specialist, question, 'clinical')

supervisor_tools = [delegate_to_finance, delegate_to_clinical, retrieve_policy]

supervisor = create_react_agent(
    model=llm,
    tools=supervisor_tools,
    prompt=supervisor_prompt,
    checkpointer=MemorySaver(),
)

print('Multi-agent system ready.')
print(f'  Supervisor tools: {[t.name for t in supervisor_tools]}')
print(f'  Finance specialist tools: {[t.name for t in finance_tools]}')
print(f'  Clinical specialist tools: {[t.name for t in healthcare_tools]}')

/tmp/ipykernel_1158/1895257894.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  _finance_specialist = create_react_agent(
/tmp/ipykernel_1158/1895257894.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  _clinical_specialist = create_react_agent(


Multi-agent system ready.
  Supervisor tools: ['delegate_to_finance', 'delegate_to_clinical', 'retrieve_policy']
  Finance specialist tools: ['get_stock_price', 'get_interest_rate', 'calculate_portfolio_value']
  Clinical specialist tools: ['check_drug_interaction', 'get_clinical_guideline']


/tmp/ipykernel_1158/1895257894.py:50: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor = create_react_agent(


In [10]:
def full_pipeline(query: str, caller_access_level: str = 'public') -> dict:
    """
    Six-stage pipeline:
    1. Input guardrail
    2. Supervisor invocation
    3. Specialist delegation (finance / clinical / RAG)
    4. Supervisor synthesis
    5. Output guardrail
    6. Audit log emission
    Returns dict with answer, thread_id, and audit record.
    """
    global CALLER_ACCESS_LEVEL
    CALLER_ACCESS_LEVEL = caller_access_level

    # ── Stage 1: Input guardrail ───────────────────────────────────────────────
    chk = input_guardrail(query)
    if not chk.passed:
        print(f'  [INPUT GUARDRAIL BLOCKED] {chk.reason}')
        return {'answer': f'[BLOCKED] {chk.reason}', 'thread_id': None, 'record': None}

    # ── Stage 2-4: Supervisor + specialists ────────────────────────────────────
    tid    = str(uuid.uuid4())
    cfg    = {'configurable': {'thread_id': tid}, 'recursion_limit': 15}
    t0     = time.monotonic()

    result   = supervisor.invoke({'messages': [HumanMessage(content=query)]}, config=cfg)
    messages = result['messages']
    answer   = messages[-1].content
    latency  = (time.monotonic() - t0) * 1000

    # ── Stage 5: Output guardrail ──────────────────────────────────────────────
    answer = output_guardrail(answer, 'combined')

    # ── Stage 6: Audit log ─────────────────────────────────────────────────────
    total_in, total_out, tool_names = 0, 0, []
    for msg in messages:
        meta  = getattr(msg, 'response_metadata', {}) or {}
        usage = meta.get('token_usage', {})
        total_in  += usage.get('prompt_tokens', 0)
        total_out += usage.get('completion_tokens', 0)
        for tc in getattr(msg, 'tool_calls', []) or []:
            tool_names.append(tc['name'])

    record = AgentTurnRecord(
        thread_id       = tid,
        domain          = 'combined',
        query_hash      = hashlib.sha256(query.encode()).hexdigest()[:16],
        input_tokens    = total_in,
        output_tokens   = total_out,
        tool_calls      = tool_names,
        latency_ms      = round(latency, 1),
        recursion_depth = len(messages),
        status          = 'ok',
    )
    _emit_audit(record)

    return {'answer': answer, 'thread_id': tid, 'record': record}


print('full_pipeline() ready.')
print('Stages: input_guardrail → supervisor → specialists → output_guardrail → audit_log')

full_pipeline() ready.
Stages: input_guardrail → supervisor → specialists → output_guardrail → audit_log


In [11]:
DIVIDER = '=' * 70

# ── Demo 1: Cross-domain investment analysis ──────────────────────────────────
print(DIVIDER)
print('DEMO 1: Cross-domain — finance + clinical + RAG policy')
print(DIVIDER)
r1 = full_pipeline(
    'Evaluate Pfizer (PFE) as an investment. Give me the current stock price, '
    'explain the Basel III capital context for banks financing biotech, '
    'and assess whether the metformin-ibuprofen interaction risk is material to their diabetes pipeline.'
)
print('\nFINAL ANSWER:')
print(r1['answer'])
rec1 = r1['record']
print(f'\nAUDIT: tools={rec1.tool_calls} | tokens={rec1.input_tokens}in/{rec1.output_tokens}out | latency={rec1.latency_ms}ms')

# ── Demo 2: RAG-only regulatory query ────────────────────────────────────────
print('\n' + DIVIDER)
print('DEMO 2: Regulatory query — RAG retrieval only')
print(DIVIDER)
r2 = full_pipeline(
    'What is the minimum CET1 capital ratio under Basel III, '
    'and what does HIPAA require for ePHI audit controls?'
)
print('\nFINAL ANSWER:')
print(r2['answer'])

# ── Demo 3: Guardrail block ────────────────────────────────────────────────────
print('\n' + DIVIDER)
print('DEMO 3: Input guardrail — PII block')
print(DIVIDER)
r3 = full_pipeline('My SSN is 123-45-6789. What stocks should I buy?')
print('Result:', r3['answer'])

DEMO 1: Cross-domain — finance + clinical + RAG policy
    [finance specialist] 4 steps, 54 chars
    [clinical specialist] 4 steps, 416 chars

FINAL ANSWER:
### Investment Evaluation of Pfizer (PFE)

#### Financial Analysis
- **Current Stock Price**: Pfizer's (PFE) stock is currently trading at **$29.54 USD**.
- **Basel III Capital Context for Biotech Financing**: Basel III regulations require banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets, with an effective CET1 floor of 7.0% when including the capital conservation buffer. Systemically important banks face additional surcharges of 1-3.5%. These stringent capital requirements can limit the ability of banks to extend credit to high-risk sectors like biotech, potentially impacting the financing environment for companies like Pfizer.

#### Clinical Assessment
- **Metformin-Ibuprofen Interaction Risk**: The interaction between metformin and ibuprofen is classified as moderate. NSAIDs,

In [12]:
# ── Retrieval evaluation ──────────────────────────────────────────────────────
@dataclass
class RetrievalCase:
    query:           str
    expected_source: str
    access_level:    str = 'public'

def evaluate_retrieval(cases: list, top_k: int = 3) -> dict:
    hits, rr = 0, []
    print('Retrieval evaluation:')
    print('-' * 65)
    for case in cases:
        docs    = retrieve(case.query, access_level=case.access_level, top_k=top_k)
        sources = [d.metadata['source'] for d in docs]
        hit     = case.expected_source in sources
        rank    = (sources.index(case.expected_source) + 1) if hit else None
        hits   += int(hit)
        rr.append(1 / rank if rank else 0)
        status = f'HIT  rank={rank}' if hit else 'MISS'
        print(f'  [{status:<12}] {case.query[:55]:<55} expected={case.expected_source}')
    n = len(cases)
    print(f'\nHit rate @{top_k}: {hits}/{n} = {hits/n:.0%}   MRR: {sum(rr)/n:.3f}')
    return {'hit_rate': hits/n, 'mrr': sum(rr)/n}

retrieval_cases = [
    RetrievalCase('minimum CET1 capital ratio',                     'basel3_capital_rules'),
    RetrievalCase('liquidity coverage ratio stress test',            'dodd_frank_165'),
    RetrievalCase('ePHI access controls audit requirements',         'hipaa_security_rule'),
    RetrievalCase('HbA1c target type 2 diabetes first line',         'ada_2024_standards'),
    RetrievalCase('CHA2DS2-VASc anticoagulation atrial fibrillation','esc_2020_af'),
    RetrievalCase('pharmaceutical trade approval CCO',               'internal_policy_IB-2024-03', 'restricted'),
]
retrieval_metrics = evaluate_retrieval(retrieval_cases)

# ── End-to-end system evaluation ──────────────────────────────────────────────
print('\n' + '=' * 65)
print('System evaluation (golden dataset):')
print('-' * 65)

@dataclass
class SystemCase:
    query:          str
    must_contain:   list
    must_not_contain: list = field(default_factory=list)

system_cases = [
    SystemCase(
        query='What is the current price of AAPL?',
        must_contain=['189'],
    ),
    SystemCase(
        query='What is the Fed Funds rate?',
        must_contain=['5.25'],
    ),
    SystemCase(
        query='Is there an interaction between warfarin and aspirin?',
        must_contain=['major', 'bleeding'],
    ),
    SystemCase(
        query='What does the ADA 2024 guideline say about type 2 diabetes?',
        must_contain=['metformin', 'hba1c'],
    ),
]

passed, failed = 0, []
for case in system_cases:
    out    = full_pipeline(case.query)
    answer = out['answer'].lower()
    ok = (
        all(s.lower() in answer for s in case.must_contain) and
        not any(s.lower() in answer for s in case.must_not_contain)
    )
    passed += int(ok)
    status  = 'PASS' if ok else 'FAIL'
    print(f'  [{status}] {case.query[:60]}')
    if not ok:
        missing = [s for s in case.must_contain if s.lower() not in answer]
        failed.append({'query': case.query[:50], 'missing': missing})

n = len(system_cases)
print(f'\nSystem eval: {passed}/{n} passed ({100*passed//n}%)')
if failed:
    print('Failed:', failed)

Retrieval evaluation:
-----------------------------------------------------------------
  [HIT  rank=1 ] minimum CET1 capital ratio                              expected=basel3_capital_rules
  [HIT  rank=1 ] liquidity coverage ratio stress test                    expected=dodd_frank_165
  [HIT  rank=1 ] ePHI access controls audit requirements                 expected=hipaa_security_rule
  [HIT  rank=1 ] HbA1c target type 2 diabetes first line                 expected=ada_2024_standards
  [HIT  rank=1 ] CHA2DS2-VASc anticoagulation atrial fibrillation        expected=esc_2020_af
  [HIT  rank=1 ] pharmaceutical trade approval CCO                       expected=internal_policy_IB-2024-03

Hit rate @3: 6/6 = 100%   MRR: 1.000

System evaluation (golden dataset):
-----------------------------------------------------------------
    [finance specialist] 4 steps, 54 chars
  [PASS] What is the current price of AAPL?
    [finance specialist] 4 steps, 40 chars
  [PASS] What is the Fed Funds rate

## Summary

This lab assembled six components into a governed, observable, end-to-end system.

**RAG layer (Cells 4–5):** Documents written to `data/` as `.txt` + `.json` sidecar pairs. `RecursiveCharacterTextSplitter` at 400 chars / 10% overlap. FAISS index built with local `sentence-transformers/all-MiniLM-L6-v2` embeddings — no Azure embedding deployment required. `retrieve()` applies client-side access control and keyword-overlap re-ranking. Production swap: replace `FAISS.from_documents` with `AzureSearch(...)` — the `retrieve_policy` tool interface is unchanged.

**Tools (Cell 7):** Finance tools (price, rate, portfolio), healthcare tools (drug interaction, clinical guideline), and the `retrieve_policy` RAG tool all follow the same `@tool` decorator pattern. Each tool is independently testable and MCP-compatible.

**Guardrails (Cell 8):** Input guardrail blocks PII patterns (SSN, card numbers, MRN) and prompt injection before any agent code runs. Output guardrail enforces finance and clinical disclaimers. Both are plain Python functions — no graph modifications required.

**Multi-agent system (Cell 9):** Supervisor holds three tools: `delegate_to_finance`, `delegate_to_clinical`, `retrieve_policy`. Specialists receive only their domain tools (least-privilege). Each specialist runs in its own thread with its own `MemorySaver` checkpointer — fully isolated.

**Full pipeline (Cell 10):** `full_pipeline()` chains all six stages: input guardrail → supervisor → specialists → output guardrail → audit log. A single function call exercises the entire system.

**Capstone demos (Cell 11):** Three end-to-end demonstrations: cross-domain investment analysis (finance + clinical + RAG), regulatory-only RAG query, and a guardrail block on a PII-containing query.

**Evaluation (Cell 12):** Retrieval evaluation measures hit rate @3 and MRR over a six-case golden dataset. System evaluation runs four queries against `must_contain` assertions and reports pass/fail. Run both after every code or configuration change before promoting to production.